In [ ]:
#!pip install "pymilvus[model]"
#!pip install langchain-milvus peft

In [9]:
## following Milvus documentation https://milvus.io/docs/insert-update-delete.md
from pymilvus import MilvusClient
from qwen3embedding import Qwen3EmbeddingModel
import numpy as np

collection_name= "SovAI" # <-- FILL_IN_Name_of_Your_Collection

## read from saved collection 
client = MilvusClient(
    uri="http://localhost:19530"
    )

embedding=Qwen3EmbeddingModel()

#client.describe_collection(COLLECTION_NAME)
def search_collection(query, topk=2, client=client, embedding=embedding):    
    COLLECTION_NAME = collection_name
    EMBEDDING_DIM=1024
    query_embeddings = embedding.get_embeddings(query, prompt_name="query")
    query_embeddings = np.float32(query_embeddings)        
    OUTPUT_FIELDS = ['id', 'chunk', 'source']    
    search_params = {"metric_type": "IP","params":{"nprobe":1024}}
    TOP_K = topk    
    results = client.search(
        COLLECTION_NAME,
        data=[query_embeddings],
        limit=TOP_K,
        search_params=search_params,
        output_fields=OUTPUT_FIELDS,
        consistency_level="Eventually")
    return results




In [14]:
query="förklarar svenskautbildningssystemet för mig, tack"
results=search_collection(query, topk=3)


embed a query


In [18]:
relevent_chunks=[f"quote_from_doc: {r[0]["entity"]["chunk"]}, sources: {r[0]["entity"]["source"]}" for r in results]
relevent_chunks="relevent_chunks:" + '\n'.join(relevent_chunks)

In [12]:
results[0][0]["entity"]["chunk"]

'Det svenska utbildningssystemet är uppbyggt i flera steg och inkluderar obligatoriska och  \nfrivilliga utbildningar. Den grundläggande utbildningen omfattar förskoleklass och grundskola,  \nsom är obligatoriska för alla barn och ungdomar. Efter grundskolan finns det ett valfritt  \ngymnasium, följt av högre studier på universitet, högskola eller yrkeshögskola.  \nUtbildningssystemet i korthet:  \n- Förskola: Rätt att gå för barn mellan 1 och 5 år.  \n- Förskoleklass: Obligatorisk från det år barnet fyller sex år.  \n- Grundskola: Obligatorisk och omfattar 9 årskurser.  \n- Gymnasium: Frivillig utbildning efter grundskolan, som bereder för högre studier.  \n- Högskola/Universitet/Yrkeshögskola: Högre utbildning för vuxna.  \n- Sfi (Svenska för invandrare): Utbildning för invandrare som vill lära sig svenska.  \n- Komvux (Kommunal vuxenutbildning): Utbildning för vuxna på grundläggande och  \n- gymnasial nivå.'

## start constructing the origianl report generation structure in langGraph node by node

In [12]:
import getpass
import os

# del os.environ['NVIDIA_API_KEY']  ## delete key and reset
if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("Valid NVIDIA_API_KEY already in environment. Delete to reset")
else:
    nvapi_key = getpass.getpass("NVAPI Key (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
global nvapi_key

Valid NVIDIA_API_KEY already in environment. Delete to reset


In [83]:

topic = "Ge en översikt över funktioner och specifika användningsfallsexempel för dessa processorenheter: CPU, GPU."
# Structure
report_organization = """Denna rapporttyp fokuserar på jämförande analys.

Rapportstrukturen bör innehålla:
1. Introduktion (ingen forskning behövs)
- Kort översikt över ämnesområdet
- Kontext för jämförelsen

2. Huvuddelsavsnitt:
- Ett dedikerat avsnitt för VARJE erbjudande som jämförs i den användardefinierade listan
- Varje avsnitt bör undersöka:
- Kärnfunktioner (punktlista)
- Arkitektur och implementering (2-3 meningar)
- Ett exempel på användningsfall (2-3 meningar)

3. Inga huvuddelsavsnitt utöver de som är dedikerade till varje erbjudande i den användardefinierade listan

4. Slutsats med jämförelsetabell (ingen forskning behövs)
- Strukturerad jämförelsetabell som:
* Jämför alla erbjudanden från den användardefinierade listan över viktiga dimensioner
* Belyser relativa styrkor och svagheter
- Slutgiltiga rekommendationer"""
number_of_queries=10


In [84]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
llm = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.environ["NVIDIA_API_KEY"]
)


def queries_producer(llm=llm, topic='', report_organization='', number_of_queries=10):   
    report_planner_query_writer_instructions="""You are an expert technical writer, helping to plan a report. 

    The report will be focused on the following topic:
    
    {topic}
    
    The report structure will follow these guidelines:
    
    {report_organization}
    
    Your goal is to generate {number_of_queries} search queries that will help gather comprehensive information for planning the report sections. 
    
    The query should:
    
    1. Be related to the topic 
    2. Help satisfy the requirements specified in the report organization
    3. MUST be in the same language as the {topic}, for example if the topic is in Swedish, then the queries should also be in Swedish
    
    Make the query specific enough to find high-quality, relevant sources while covering the breadth needed for the report structure.
    You MUST return the queries as strings in a list: 
    below is an example
    ["query_1", "query_2", ...]
    Begin!""".format(topic=topic, report_organization=report_organization, number_of_queries=number_of_queries)
    sys_prompt="detailed thinking on, "+ system_prompt
    
    completion = llm.chat.completions.create(
      model="nvidia/llama-3.3-nemotron-super-49b-v1",
      messages=[{"role":"system","content":sys_prompt}],
      temperature=0.6,
      top_p=0.95,
      max_tokens=4096,
      frequency_penalty=0,
      presence_penalty=0,
      stream=False
    )
    output= completion.choices[0].message.content
    if '[' in output:
        start_index=output.index('[')
        list_in_string_format=output[start_index:]
        ls=json.loads(list_in_string_format)
        return True, ls
    else:
        ls=[]
        return False, ls

In [85]:
succeed_flag, output=queries_producer(llm=llm, topic='', report_organization='', number_of_queries=10)
output

['CPU vs GPU: översikt över kärnfunktioner och arkitektur',
 'CPU-arkitektur: pipeline, cacheminne och klockfrekvens',
 'GPU-arkitektur: CUDA-kärnor, strålföljd och minnesbuss',
 'Exempel på CPU-användning: servrar, embedded system och batchbearbetning',
 'Exempel på GPU-användning: grafikrendering, AI-trafik och kryptobråkning',
 'Jämförelse CPU-GPU: prestanda per watt i olika arbetsbelastningar',
 'CPU-styrkor och svagheter: single-threadad bearbetning och latens',
 'GPU-styrkor och svakheter: parallellbearbetning och kostnadseffektivitet',
 'Rekommendationer för CPU/GPU-val: specifika krav för olika branscher',
 'CPU-GPU-samarbete: exempel på hybridarkitektur och optimeringstekniker']

In [77]:
# Prompt generating the report outline
report_planner_instructions="""You are an expert technical writer, helping to plan a report.

Your goal is to generate the outline of the sections of the report. 

The overall topic of the report is:

{topic}

The report should follow this organization: 

{report_organization}

You should reflect on this information to plan the sections of the report: 

{context}

Now, generate the sections of the report. Each section should have the following fields:

- Name - Name for this section of the report.
- Description - Brief overview of the main topics and concepts to be covered in this section.
- Research - Whether to perform web research for this section of the report.
- Content - The content of the section, which you will leave blank for now.

Consider which sections require web research. For example, introduction and conclusion will not require research because they will distill information from other parts of the report."""



'["CPU core features list for technical specifications",  \n"GPU core features for parallel computing applications",  \n"CPU microarchitecture design and pipeline explanation",  \n"GPU architecture overview for massively parallel processing",  \n"CPU use case examples in single-threaded applications",  \n"GPU use case examples in deep learning model training",  \n"CPU vs GPU comparison table key dimensions for processing",  \n"Relative strengths and weaknesses of CPU vs GPU in workload handling",  \n"CPU use case scenarios in embedded systems and IoT devices",  \n"GPU use case examples in scientific simulation and graphics rendering"]'

In [79]:
ls

['CPU core features list for technical specifications',
 'GPU core features for parallel computing applications',
 'CPU microarchitecture design and pipeline explanation',
 'GPU architecture overview for massively parallel processing',
 'CPU use case examples in single-threaded applications',
 'GPU use case examples in deep learning model training',
 'CPU vs GPU comparison table key dimensions for processing',
 'Relative strengths and weaknesses of CPU vs GPU in workload handling',
 'CPU use case scenarios in embedded systems and IoT devices',
 'GPU use case examples in scientific simulation and graphics rendering']

In [70]:
if "</think>" in output :
    
    list_in_string_format=output.split("</think>")[1]
    list_in_string_format

In [59]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
llm = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.environ["NVIDIA_API_KEY"]
)

def retriver_with_llm_response(llm=llm, input_query='' ):
    results=search_collection(input_query, topk=3)
    relevent_info=[f"quote_from_doc: {r[0]["entity"]["chunk"]}, sources: {r[0]["entity"]["source"]}" for r in results]
    relevent_chunks="relevent_chunks:" + '\n'.join(relevent_info)
    sys_prompt = f"""detailed thinking on, You are a helpful assistant and an expert who is excellent in answering incoming user query grounding your answer to the relevant document database which you have access to.
    user input query : {input_query}
    relevent_chunks : {relevent_chunks}
        
    You should STRICTLY follow the below format when you construct a plan
    - put detai thinking on ONLY when it requires you to integrate complicated and long context from multiple sources, if it is a simple question 
    - You should integrate the relevent information retrieved from the database and ground your response to the relevent information 
    - You MUST answer in the same langauge as input user query
    - You should rephrase multiple retrieved information into a condensed summary but make sure you stay truthful to the original content
    - If the retrieved information is irrelevant to the user query, then make sure you tell the user you cannot answer this question because you do not have sufficient information
    - You must return the response in the JSON format as shown below :
    {{"detailed_thinking": "on" , "response": "XXX" , "reference": "sources referenced in the relevent_chunks" }}\n\n    
    """
    
    completion = llm.chat.completions.create(
      model="nvidia/llama-3.3-nemotron-super-49b-v1",
      messages=[{"role":"system","content":sys_prompt},{"role":"user","content":input_query}],
      temperature=0.6,
      top_p=0.95,
      max_tokens=4096,
      frequency_penalty=0,
      presence_penalty=0,
      stream=False
    )
    return completion.choices[0].message.content
query = "förklarar svenskautbildningssystemet för mig, tack"
response=LLM_response(input_query=query)


embed a query


In [44]:
response

'<think>\nOkay, the user wants an explanation of the Swedish education system. Let me check the relevant chunks provided. The document mentions different stages: förskola, förskoleklass, grundskola, gymnasium, högskola/universitet/yrkeshögskola, Sfi, and Komvux. I need to structure this in a clear way.\n\nFirst, start with the overview from the chunk. Then break down each level. Förskola is for 1-5 year olds, not mandatory. Förskoleklass is compulsory from age 6. Grundskola is 9 years, mandatory. Then gymnasium is optional, preparing for higher education. After that, högskola, universitet, or yrkeshögskola for higher studies. Also, Sfi for immigrants learning Swedish and Komvux for adult education.\n\nMake sure to use Swedish terms as the user asked in Swedish. Keep it concise but cover all points. Check if all parts of the chunk are included. Avoid adding extra info not in the source. Rephrase for clarity without changing meaning. Structure with bullet points or short paragraphs for r

In [51]:
import json

def parse_to_response(response):
    text = response
    
    if '</think>' in text:
        raw_plan_str = text.split('</think>')[1]
        #print("raw_plan_str=\n", raw_plan_str)
    else:
        raw_plan_str = text
        #print("raw_plan_str=\n", raw_plan_str)
    try:
        
        new_plan=json.loads(raw_plan_str)
        return True, new_plan
    
    except Exception:
        print("ERROR in parse_to_list_of_steps")
        return False, ''
    

In [52]:
flag, out=parse_to_response(response)
out.items()

raw_plan_str=
 

{"detailed_thinking": "off" , "response": "Svenskt utbildningssystem består av flera steg, både obligatoriska och frivilliga. Här är en översikt:\n- **Förskola**: För barn 1–5 år (ej obligatorisk).\n- **Förskoleklass**: Obligatorisk från 6 års ålder.\n- **Grundskola**: Obligatorisk, 9 årskurser.\n- **Gymnasium**: Frivilligt, bereder för högre studier.\n- **Högskola/Universitet/Yrkeshögskola**: Högre utbildning för vuxna.\n- **Sfi (Svenska för invandrare)**: För invandrare som vill lära sig svenska.\n- **Komvux (Kommunal vuxenutbildning)**: Utbildning för vuxna på grundläggande och gymnasial nivå.", "reference": "doc:swedish_education_system.md > Header_1:Svenska Utbildningssystemet > chunk_id_5"}


dict_items([('detailed_thinking', 'off'), ('response', 'Svenskt utbildningssystem består av flera steg, både obligatoriska och frivilliga. Här är en översikt:\n- **Förskola**: För barn 1–5 år (ej obligatorisk).\n- **Förskoleklass**: Obligatorisk från 6 års ålder.\n- **Grundskola**: Obligatorisk, 9 årskurser.\n- **Gymnasium**: Frivilligt, bereder för högre studier.\n- **Högskola/Universitet/Yrkeshögskola**: Högre utbildning för vuxna.\n- **Sfi (Svenska för invandrare)**: För invandrare som vill lära sig svenska.\n- **Komvux (Kommunal vuxenutbildning)**: Utbildning för vuxna på grundläggande och gymnasial nivå.'), ('reference', 'doc:swedish_education_system.md > Header_1:Svenska Utbildningssystemet > chunk_id_5')])